# Random Variables & Distributions

## What's covered

- What a **random variable** actually is — a function from outcomes to numbers
- **Discrete vs continuous** — the two flavors and why they need different machinery
- **PMF, PDF, CDF** — the three functions that describe a distribution
- The staple discrete distributions — **Bernoulli, Binomial, Categorical, Multinomial, Poisson, Geometric**
- The staple continuous distributions — **Uniform, Exponential, Gaussian** (preview), **Beta, Gamma**
- **The exponential family** — the meta-pattern most ML distributions belong to


## What is a random variable?

The English word "variable" is misleading. A **random variable** is *not* a variable — it's a **function**. Specifically, it's a function from the sample space `\Omega` to the real numbers:

$$
X : \Omega \to \mathbb{R}
$$

For a coin flip with `\Omega = \{H, T\}`, the indicator random variable `X(H) = 1`, `X(T) = 0` turns the outcome into a number we can do arithmetic on. For two rolls of a die, `X = \text{(first roll)} + \text{(second roll)}` is a random variable taking values in `\{2, 3, \dots, 12\}`. Both inputs are abstract outcomes; the random variable extracts a numeric summary.

**Why bother?** Because once outcomes are numbers, we can compute averages, variances, and run them through more functions. *Every* probabilistic model in ML is built on random variables — features `X`, labels `Y`, parameters `\theta`, hidden states `Z`, all of them.

**Convention.** Uppercase letters (`X`, `Y`, `Z`) for random variables. Lowercase (`x`, `y`, `z`) for specific *values* the random variable might take. So `P(X = x)` reads "the probability that the random variable `X` takes the value `x`."

**Two flavors.** A random variable is **discrete** if it can only take values in a finite or countable set (`\{0, 1\}`, `\{1, 2, 3, \dots\}`, words in a vocabulary). It's **continuous** if it ranges over an interval of the real line. These two flavors need slightly different machinery, but the ideas line up.


## PMF, PDF, and CDF — describing a distribution

A random variable doesn't really *exist* in isolation; it has a **distribution**, which is a description of how its values are spread. Three functions describe a distribution, depending on flavor.

**Probability mass function (PMF)** — for discrete `X`. The PMF is

$$
p_X(x) = P(X = x)
$$

PMF values are *actual probabilities*. They sum to 1 over all `x`. For a fair die, `p_X(k) = 1/6` for `k = 1, \dots, 6`.

**Probability density function (PDF)** — for continuous `X`. The PDF `f_X(x)` is *not* a probability. `P(X = x) = 0` for any specific value, because a continuous variable has infinitely many possible values. Instead the PDF describes probability *per unit interval*:

$$
P(a \le X \le b) = \int_a^b f_X(x) \, dx
$$

The PDF integrates to 1 over its support. It can be greater than 1 at any point — it's a density, not a probability. The familiar bell curve `f(x) = (1/\sqrt{2\pi}) e^{-x^2/2}` is the standard normal PDF; it peaks at `1/\sqrt{2\pi} \approx 0.4`, not 1.

**Cumulative distribution function (CDF)** — for *both* discrete and continuous. The CDF is

$$
F_X(x) = P(X \le x)
$$

The CDF starts at 0, monotonically increases to 1, and ties together the discrete and continuous cases — useful for sampling, for tail probabilities, for testing.

**Relationships.** For continuous `X`, `F_X(x) = \int_{-\infty}^x f_X(t) \, dt` and (by the fundamental theorem of calculus from the calculus repo) `f_X(x) = F_X'(x)`. For discrete `X`, `F_X(x) = \sum_{k \le x} p_X(k)` — the CDF jumps at each value where `p_X > 0`.

**Support.** The set of values where `p_X(x) > 0` (discrete) or `f_X(x) > 0` (continuous). The Gaussian's support is `\mathbb{R}`; the Exponential's is `[0, \infty)`; the Bernoulli's is `\{0, 1\}`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

# Side-by-side: PMF of a fair die, PDF of standard normal, both with their CDFs

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

# Die PMF
xs = np.arange(1, 7)
axes[0, 0].bar(xs, [1/6] * 6)
axes[0, 0].set_title("PMF — fair die")
axes[0, 0].set_xlabel("k"); axes[0, 0].set_ylabel("P(X = k)")

# Die CDF (step function)
cdf_vals = np.cumsum([1/6] * 6)
axes[0, 1].step(xs, cdf_vals, where="post")
axes[0, 1].set_title("CDF — fair die")
axes[0, 1].set_xlabel("x"); axes[0, 1].set_ylabel("P(X ≤ x)")
axes[0, 1].set_ylim(0, 1.05)

# Normal PDF
xs_c = np.linspace(-4, 4, 400)
axes[1, 0].plot(xs_c, stats.norm.pdf(xs_c))
axes[1, 0].set_title("PDF — standard normal")
axes[1, 0].set_xlabel("x"); axes[1, 0].set_ylabel("f(x)")

# Normal CDF
axes[1, 1].plot(xs_c, stats.norm.cdf(xs_c))
axes[1, 1].set_title("CDF — standard normal")
axes[1, 1].set_xlabel("x"); axes[1, 1].set_ylabel("F(x)")

plt.tight_layout()
plt.show()
print("Note the normal PDF peaks above 0.4 — density values are not probabilities.")


## Bernoulli and Binomial

The simplest distribution. `X \sim \text{Bernoulli}(p)` takes value 1 with probability `p` and 0 with probability `1 - p`. The PMF:

$$
p_X(x) = p^x (1 - p)^{1 - x}, \quad x \in \{0, 1\}
$$

(A slick way to write "p if x=1, 1-p if x=0" in one formula.)

`X \sim \text{Binomial}(n, p)` counts the number of successes in `n` independent Bernoulli trials, each with success probability `p`. The PMF:

$$
p_X(k) = \binom{n}{k} p^k (1 - p)^{n - k}, \quad k \in \{0, 1, \dots, n\}
$$

The binomial coefficient `\binom{n}{k}` counts the number of ways to arrange `k` successes among `n` trials.

**Common confusion.** Bernoulli is the *single-trial* version; Binomial is the *count over n trials* version. Setting `n = 1` recovers Bernoulli.

**ML usage.** Binary classification labels are Bernoulli. Logistic regression *is* MLE for a Bernoulli likelihood — that's why its loss is `-\log p(y | x)`. The cross-entropy loss for binary classification, `-y \log \hat{p} - (1-y) \log (1 - \hat{p})`, is literally the negative log of the Bernoulli PMF written with `\hat{p}` as the predicted parameter.


In [ ]:
p = 0.3
n = 20

# Sampling from Bernoulli and Binomial
bernoulli_samples = rng.random(10_000) < p           # equivalent to Bernoulli(p)
binomial_samples  = rng.binomial(n, p, size=10_000)

print(f"Bernoulli({p}): mean = {bernoulli_samples.mean():.4f}   (analytic {p})")
print(f"Binomial({n}, {p}): mean = {binomial_samples.mean():.4f}   (analytic {n*p})")
print(f"Binomial({n}, {p}): var  = {binomial_samples.var():.4f}    (analytic {n*p*(1-p)})")

# PMF — analytic vs empirical
ks = np.arange(0, n + 1)
analytic = stats.binom.pmf(ks, n, p)
empirical = np.bincount(binomial_samples, minlength=n + 1) / len(binomial_samples)

print("\nBinomial PMF check:")
print(f"{'k':>3} {'analytic':>10} {'empirical':>11}")
for k in range(11):
    print(f"{k:>3} {analytic[k]:>10.4f} {empirical[k]:>11.4f}")


## Categorical and Multinomial

`X \sim \text{Cat}(\mathbf{p})` generalizes Bernoulli to `K` outcomes. `X` takes value `k \in \{1, 2, \dots, K\}` with probability `p_k`, where `\mathbf{p} = (p_1, \dots, p_K)` is a probability vector (entries non-negative, summing to 1). PMF:

$$
p_X(k) = p_k
$$

In code, categorical values are usually represented as **one-hot vectors** — `(0, 0, 1, 0, \dots, 0)` with the 1 in the `k`-th slot. This is what softmax outputs are intended to model.

`Y \sim \text{Multinomial}(n, \mathbf{p})` counts how many times each of the `K` outcomes appears in `n` independent categorical draws. `Y = (Y_1, \dots, Y_K)` with `\sum_k Y_k = n`. PMF:

$$
P(Y = (n_1, \dots, n_K)) = \frac{n!}{n_1! \, n_2! \, \cdots \, n_K!} \prod_{k=1}^{K} p_k^{n_k}
$$

**ML usage.** Multi-class classification: the predicted distribution over `K` classes is exactly a `Cat(\hat{\mathbf{p}})`. Cross-entropy loss against a one-hot target is the negative log-likelihood of the Categorical. Language modeling: each next-token distribution is a `Cat(\hat{\mathbf{p}})` over the vocabulary.


In [ ]:
# Categorical with 4 classes
p_vec = np.array([0.1, 0.4, 0.3, 0.2])
K = len(p_vec)

# Sample via numpy's choice
cat_samples = rng.choice(K, size=50_000, p=p_vec)
empirical = np.bincount(cat_samples, minlength=K) / len(cat_samples)
print("Categorical:")
print(f"  analytic p  = {p_vec}")
print(f"  empirical   = {empirical.round(4)}")

# Multinomial — draw 100 trials, see counts
multi_sample = rng.multinomial(100, p_vec)
print(f"\nOne Multinomial(100, p) draw: counts = {multi_sample}, total = {multi_sample.sum()}")

# Average over many multinomial draws → each component's mean is n * p_k
n_trials = 20
draws = np.array([rng.multinomial(n_trials, p_vec) for _ in range(20_000)])
print(f"\nMean counts per class after Multinomial({n_trials}, p): {draws.mean(axis=0).round(3)}")
print(f"Analytic   n * p:                                  {(n_trials * p_vec).round(3)}")


## Poisson and Geometric

**Poisson** `X \sim \text{Poisson}(\lambda)` counts the number of events in a fixed time window when events happen independently at average rate `\lambda`. Support is `\{0, 1, 2, \dots\}`. PMF:

$$
p_X(k) = \frac{\lambda^k e^{-\lambda}}{k!}
$$

Mean and variance both equal `\lambda`. The "events are rare but many opportunities" limit of the Binomial — as `n \to \infty`, `p \to 0`, `np \to \lambda`, the Binomial converges to a Poisson.

**ML usage.** Count data — word counts, click-through events, fraud signals — are modeled as Poisson. Poisson regression generalizes linear regression to count targets. Some attention variants use Poisson processes for sparse attention patterns.

**Geometric** `X \sim \text{Geometric}(p)` counts the number of trials *until the first success* in a sequence of Bernoulli(`p`) trials. Support `\{1, 2, 3, \dots\}`, PMF:

$$
p_X(k) = (1 - p)^{k - 1} p
$$

Mean `1/p`. The discrete analogue of the Exponential distribution, which it converges to in the small-time-step limit.

**ML usage.** Modeling time-to-failure for systems, RL with discount factor `\gamma` (a geometric distribution implicitly), early-stopping analysis.


In [ ]:
lam = 3.0
poisson_samples = rng.poisson(lam, size=10_000)
print(f"Poisson({lam}): mean = {poisson_samples.mean():.4f}, var = {poisson_samples.var():.4f}  (both should be ~{lam})")

# PMF comparison
ks = np.arange(0, 12)
print(f"\n{'k':>3} {'analytic':>10} {'empirical':>11}")
for k in ks:
    a = stats.poisson.pmf(k, lam)
    e = np.mean(poisson_samples == k)
    print(f"{k:>3} {a:>10.4f} {e:>11.4f}")

# Geometric (numpy's geometric: P(X = k) = (1 - p)^(k-1) * p for k = 1, 2, 3, ...)
p = 0.25
geom_samples = rng.geometric(p, size=10_000)
print(f"\nGeometric({p}): mean = {geom_samples.mean():.4f}  (analytic {1/p})")


## Continuous staples — Uniform, Exponential, Gaussian, Beta, Gamma

**Uniform** `X \sim \mathcal{U}(a, b)`. Constant density on `[a, b]`, zero outside:

$$
f_X(x) = \frac{1}{b - a}, \quad x \in [a, b]
$$

Mean `(a+b)/2`, variance `(b-a)^2 / 12`. The flat distribution — your "I have no idea where it is in this range" prior.

**Exponential** `X \sim \text{Exp}(\lambda)`. Models waiting time between independent events arriving at rate `\lambda`. Support `[0, \infty)`:

$$
f_X(x) = \lambda e^{-\lambda x}, \quad x \ge 0
$$

Mean `1/\lambda`, variance `1/\lambda^2`. **Memoryless** — having already waited `t` time units doesn't change the distribution of remaining wait. (We'll show this in the next notebook.)

**Gaussian** `X \sim \mathcal{N}(\mu, \sigma^2)`. The bell curve:

$$
f_X(x) = \frac{1}{\sigma \sqrt{2 \pi}} \exp\left(-\frac{(x - \mu)^2}{2 \sigma^2}\right)
$$

Mean `\mu`, variance `\sigma^2`. **The single most important distribution in ML.** Notebook 5 is dedicated to it.

**Beta** `X \sim \text{Beta}(\alpha, \beta)`. Support `[0, 1]`. PDF (proportional to):

$$
f_X(x) \propto x^{\alpha - 1} (1 - x)^{\beta - 1}
$$

The natural distribution over probabilities — flexible shape controlled by `\alpha, \beta`. Conjugate to the Bernoulli; we'll use it in notebook 6.

**Gamma** `X \sim \text{Gamma}(\alpha, \beta)`. Support `[0, \infty)`. PDF (proportional to):

$$
f_X(x) \propto x^{\alpha - 1} e^{-\beta x}
$$

Generalizes the Exponential (`\alpha = 1`). Models positive-valued quantities; conjugate to the Poisson rate parameter.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

xs = np.linspace(0, 5, 400)
axes[0].plot(xs, stats.expon.pdf(xs, scale=1), label="Exp(1)")
axes[0].plot(xs, stats.expon.pdf(xs, scale=0.5), label="Exp(2)")
axes[0].set_title("Exponential densities"); axes[0].legend()

xs = np.linspace(0, 1, 400)
for a, b in [(0.5, 0.5), (2, 5), (5, 1), (2, 2)]:
    axes[1].plot(xs, stats.beta.pdf(xs, a, b), label=f"Beta({a}, {b})")
axes[1].set_title("Beta densities"); axes[1].legend(fontsize=8)

xs = np.linspace(0, 12, 400)
for a, b in [(1, 1), (2, 1), (5, 1), (9, 2)]:
    axes[2].plot(xs, stats.gamma.pdf(xs, a, scale=1/b), label=f"Gamma({a}, {b})")
axes[2].set_title("Gamma densities"); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()
print("Beta is a shape-flexible distribution on [0, 1] — great for modeling probabilities.")
print("Gamma generalizes Exponential to model positive quantities with heavier or lighter tails.")


## The exponential family — the meta-pattern

Most of the distributions above (Bernoulli, Binomial, Poisson, Gaussian, Beta, Gamma, Exponential, Categorical, Dirichlet, …) share a common shape:

$$
p(x | \boldsymbol{\eta}) = h(x) \exp\!\bigl(\boldsymbol{\eta}^\top T(x) - A(\boldsymbol{\eta})\bigr)
$$

where `\boldsymbol{\eta}` are the **natural parameters**, `T(x)` are the **sufficient statistics**, `h(x)` is the **base measure**, and `A(\boldsymbol{\eta})` is the **log-partition function** that makes the density normalize.

This is the **exponential family**, and it's where probability gets really beautiful. Key properties (proved in any Bayesian textbook):

- The **sufficient statistic** `T(x)` summarizes the data perfectly — no information is lost by replacing `x_1, \dots, x_n` with `\sum_i T(x_i)`.
- Each exponential-family distribution has a **conjugate prior** that's also exponential-family. We'll see Beta-Bernoulli and Gaussian-Gaussian conjugacy in notebook 6.
- The **gradient of the log-partition** is the expected sufficient statistic: `\nabla A(\boldsymbol{\eta}) = \mathbb{E}[T(X)]`. This identity is the secret sauce of variational inference.
- **Maximum likelihood is easy** — gradient of log-likelihood involves only the empirical sufficient statistics.

In practice you don't need to chant the exponential-family form every time. But knowing the framework explains why so many distributions feel "related" — they really are.

**Generalized Linear Models (GLMs)** are the bridge to ML: linear regression (Gaussian), logistic regression (Bernoulli), Poisson regression (Poisson), softmax regression (Categorical) — all GLMs with different exponential-family choices. The link functions are inverses of the canonical parameter mappings.


## Where this appears in ML

Distributions are the building blocks of probabilistic ML. Most architectures correspond to a choice of distribution for outputs and (sometimes) for latents.

- **Logistic regression / binary cross-entropy.** Output is `\text{Bernoulli}(\hat{p})`; loss is `-\log p(y | x)`.
- **Softmax classification / cross-entropy.** Output is `\text{Cat}(\hat{\mathbf{p}})`; loss is again negative log-likelihood.
- **Regression with MSE.** Output is `\mathcal{N}(\hat{y}, \sigma^2)`; the MSE loss *is* the negative log-likelihood (up to constants).
- **Poisson regression for counts.** Use a Poisson output distribution with `\log \lambda = \mathbf{w}^\top \mathbf{x}`.
- **VAEs.** Encoder outputs a Gaussian `\mathcal{N}(\mu_\phi(x), \sigma^2_\phi(x))` over the latent; decoder outputs Bernoulli (images) or Gaussian (continuous data).
- **Diffusion models.** Each denoising step is a (potentially Gaussian) conditional distribution over the slightly less-noisy version.
- **Language models.** Each next-token distribution is `\text{Cat}(\hat{\mathbf{p}})` over the vocabulary. The "temperature" in sampling literally rescales the natural parameters.
- **Latent Dirichlet Allocation (LDA).** Documents are mixtures of topics drawn from Dirichlets; words are drawn from Categoricals.
- **Gaussian processes.** The output at every input is jointly Gaussian — leans on the multivariate Gaussian for everything.
- **Mixture models (GMMs).** Probabilistic clustering with weighted sums of distributions, usually Gaussian.
- **Bayesian neural networks.** Treat weights as random variables; specify priors (often Gaussian); compute (approximate) posteriors.
- **Reinforcement learning.** Policies are distributions over actions; rewards have distributions; value functions are expectations.

Next notebook: **expectation, variance, and moments** — how we summarize distributions with a few key numbers, and the identities (linearity, LOTUS, covariance) you'll lean on for the rest of the repo.
